In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Wedge
from IPython.display import HTML

# --------- SETTINGS ----------
value_percent = 47.1          # fill up to this (0..100)
color_fill = "#467592"
color_bg = "#E6E6E6"

fps = 30
animate_duration_sec = 3
pause_duration_sec = 2
total_frames = int(fps * animate_duration_sec)
pause_frames = int(fps * pause_duration_sec)
frames_total = total_frames + pause_frames

# --------- FIGURE ----------
fig, ax = plt.subplots(figsize=(8, 4.8))
fig.patch.set_facecolor("#ffffff")

def draw_gauge(pct):
    ax.clear()
    ax.set_aspect("equal")
    ax.axis("off")

    # Gauge geometry
    R = 1.0          # outer radius
    width = 0.25     # thickness of the arc

    # Map percent (0..100) to angle span (180..0)
    # left end = 180°, right end = 0°
    end_angle = 180 - (180 * (pct / 100.0))

    # Background semicircle
    bg = Wedge((0, 0), R, 0, 180, width=width, facecolor=color_bg, edgecolor="none")
    ax.add_patch(bg)

    # Filled portion (from left towards right)
    fill = Wedge((0, 0), R, end_angle, 180, width=width, facecolor=color_fill, edgecolor="none")
    ax.add_patch(fill)

    # Center text
    ax.text(0, -0.05, f"{pct:.1f}%", ha="center", va="center",
            fontsize=28, fontweight="bold", color="#222222")

    # Optional: min/max labels
    ax.text(-1.05, -0.35, "0", ha="center", va="center", fontsize=11, color="#666666")
    ax.text( 1.05, -0.35, "100", ha="center", va="center", fontsize=11, color="#666666")

    # Limits so it looks like a speedometer
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-0.6, 1.2)

def animate(frame):
    if frame >= total_frames:
        pct = value_percent
    else:
        pct = value_percent * (frame / total_frames)
    draw_gauge(pct)

anim = FuncAnimation(fig, animate, frames=frames_total, interval=1000/fps, blit=False)
plt.close(fig)

# Inline display (Jupyter)
HTML(anim.to_html5_video())

# # Optional save to mp4
# writer = FFMpegWriter(fps=fps, codec="libx264", bitrate=2500, extra_args=["-pix_fmt", "yuv420p"])
# anim.save("gauge_180.mp4", writer=writer, dpi=200)
# print("✅ Saved: gauge_180.mp4")
# Inline display (Jupyter)
HTML(anim.to_html5_video())

✅ Saved: gauge_180.mp4


In [ ]:
from google.colab import files
files.download('gauge_180.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle
from matplotlib.patches import Arc
from IPython.display import HTML

# ---------------- DATA / TEXT ----------------
title_text = ""
value = 83.8               # main gauge value (0..100)

# ---------------- STYLE ----------------
fill_color = "#B56400"     # orange/brown like screenshot
bg_color   = "#E3E3E3"     # remaining arc
tick_color = "#D8D8D8"     # inner ticks
text_dark  = "#666666"

# Geometry
R = 1.0                    # gauge radius
arc_lw = 14                # thickness of main arc (linewidth)
tick_count = 70            # number of tick marks
tick_len = 0.10            # tick length
tick_lw = 2.0              # tick thickness
tick_radius = R - 0.18     # where ticks sit

# Animation
fps = 30
animate_duration_sec = 2.5
pause_duration_sec = 1.5
total_frames = int(fps * animate_duration_sec)
pause_frames = int(fps * pause_duration_sec)
frames_total = total_frames + pause_frames

# ---------------- FIGURE ----------------
fig, ax = plt.subplots(figsize=(7, 4.5), dpi=120)
fig.patch.set_facecolor("#F2F2F2")  # outer light gray panel background
ax.set_facecolor("white")

def draw_panel_background():
    # a white "card" look
    ax.add_patch(Circle((0, 0), 0, alpha=0))  # no-op to keep patches order simple

def draw_ticks():
    # ticks from 180° (left) to 0° (right) across the top semicircle
    angles = np.linspace(180, 0, tick_count)
    for a in angles:
        th = np.deg2rad(a)
        x0, y0 = tick_radius * np.cos(th), tick_radius * np.sin(th)
        x1, y1 = (tick_radius - tick_len) * np.cos(th), (tick_radius - tick_len) * np.sin(th)
        ax.plot([x0, x1], [y0, y1], color=tick_color, lw=tick_lw, solid_capstyle="round", zorder=1)

def draw_gauge(pct):
    ax.clear()
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_facecolor("white")

    # Title
    ax.text(0, 1.12, title_text, ha="center", va="center",
            fontsize=14, fontweight="bold", color="#7A7A7A")

    # Inner tick marks
    draw_ticks()

    # Background arc (full semicircle)
    bg_arc = Arc((0, 0), 2*R, 2*R, angle=0, theta1=0, theta2=180,
                 lw=arc_lw, color=bg_color, capstyle="round", zorder=2)
    ax.add_patch(bg_arc)

    # Filled arc portion
    # Map 0..100 to 180..0 fill span
    # Fill should start at left (180°) and progress toward right (0°)
    end_angle = 180 - 180*(pct/100.0)   # 0% -> 180, 100% -> 0
    if pct > 0:
        fill_arc = Arc((0, 0), 2*R, 2*R, angle=0, theta1=end_angle, theta2=180,
                       lw=arc_lw, color=fill_color, capstyle="round", zorder=3)
        ax.add_patch(fill_arc)

    # Center white circle + soft shadow
    shadow = Circle((0.02, -0.02), 0.40, facecolor="black", edgecolor="none", alpha=0.08, zorder=3)
    ax.add_patch(shadow)
    center = Circle((0, 0), 0.40, facecolor="white", edgecolor="#F0F0F0", lw=1.0, zorder=4)
    ax.add_patch(center)

    # Center value (use comma like your screenshot)
    value_txt = f"{pct:.1f}".replace(".", ",")
    ax.text(0, -0.02, value_txt, ha="center", va="center",
            fontsize=24, fontweight="bold", color=fill_color, zorder=5)

        # Move labels slightly down + toward center
    ax.text(-1, -0.2, "0", ha="center", va="center", fontsize=12, color="#333333")
    ax.text( 1, -0.2, "100", ha="center", va="center", fontsize=12, color="#333333")



    # Limits for a nice “card” framing
    ax.set_xlim(-1.25, 1.25)
    ax.set_ylim(-0.65, 1.25)

def animate(frame):
    if frame >= total_frames:
        pct = value
    else:
        pct = value * (frame / total_frames)
    draw_gauge(pct)

anim = FuncAnimation(fig, animate, frames=frames_total, interval=1000/fps, blit=False)
plt.close(fig)

# Show in notebook:
HTML(anim.to_html5_video())

# Optional save:
# writer = FFMpegWriter(fps=fps, codec="libx264", bitrate=2500, extra_args=["-pix_fmt", "yuv420p"])
# anim.save("service_gauge.mp4", writer=writer, dpi=200)
